# Week-1 classic baseline

CPU only. In the Kaggle notebook settings set the accelerator to **None**. Do not enable the GPU.

The GitHub repository stays private. Do not make it public. Upload a ZIP of this repo (without `data/`, `.env`, or `kaggle.json`) as a **private** Kaggle dataset. The first code cell copies that dataset's `src` into `/kaggle/working/src` and puts that directory on `PYTHONPATH`.

`FILAMENT_DATA_ROOT` is the competition **test** image folder, not the sibling `train` directory:

`/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/test/test_images`

Test files are `.jpeg` (for example `20220621185332Bh.jpeg`). `segment_filaments` defaults are an untuned starting point for full-disk images on the order of 2k pixels. They are not fitted on MAGFiLO. The synthetic 64×64 demo passes a smaller `block_size` and `min_area`.

The submission path builds one label image, then `iter_filament_counts` yields `(filament_number, RLE counts)` one component at a time. It does not allocate a list of full-size masks.


In [ ]:
import os
import shutil
import sys
from pathlib import Path

import numpy as np

# Test images only. The sibling train directory is not the submission input.
TEST_IMAGES = (
    "/kaggle/input/competitions/filament-segmentation-2026/"
    "MAGFiLO_1.0_Kaggle_2026/test/test_images"
)


def _local_src() -> Path | None:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        src = candidate / "src"
        if (src / "filament" / "runtime.py").is_file():
            return src
    return None


def _private_dataset_src(root: Path | None = None) -> Path | None:
    """Return the shallowest private-dataset ``src`` under ``root``.

    ``filament/runtime.py`` may sit at any depth. Directories named
    ``competitions`` are not walked, so the MAGFiLO tree is left alone.
    """
    root = Path("/kaggle/input") if root is None else root
    if not root.is_dir():
        return None
    marker = Path("filament") / "runtime.py"
    matches: list[Path] = []
    for dirpath, dirnames, _filenames in os.walk(root, topdown=True):
        dirnames[:] = [name for name in dirnames if name != "competitions"]
        current = Path(dirpath)
        if current.name != "src" or not (current / marker).is_file():
            continue
        matches.append(current)
        dirnames.clear()
    if not matches:
        return None
    return min(
        matches,
        key=lambda path: (len(path.relative_to(root).parts), path.as_posix()),
    )


def _put_on_path(src: Path) -> None:
    text = str(src)
    if text not in sys.path:
        sys.path.insert(0, text)
    current = os.environ.get("PYTHONPATH", "")
    parts = [part for part in current.split(os.pathsep) if part]
    if text not in parts:
        os.environ["PYTHONPATH"] = os.pathsep.join([text, *parts]) if parts else text


def _prepare_src() -> None:
    """Put this repo's src on sys.path.

    Local checkouts are used as they are. On Kaggle the GitHub repo stays
    private: copy ``src`` from the private dataset into ``/kaggle/working/src``
    and put that directory on ``PYTHONPATH``.
    """
    local = _local_src()
    if local is not None:
        _put_on_path(local)
        return
    destination = Path("/kaggle/working/src")
    if (destination / "filament" / "runtime.py").is_file():
        _put_on_path(destination)
        return
    source = _private_dataset_src()
    if source is None:
        raise FileNotFoundError(
            "Could not find src/filament/runtime.py in the working "
            "directory or in a private dataset under /kaggle/input "
            "(any depth outside competitions, for example "
            "/kaggle/input/datasets/ignacioaguirrecandia/"
            "week1-solar-filament-segmentation/"
            "Solar-Filament-Segmentation-main/src). "
            "Upload a ZIP of this private repo as a private Kaggle "
            "dataset, without data/, .env, or kaggle.json. Do not make "
            "the GitHub repository public."
        )
    shutil.copytree(source, destination, dirs_exist_ok=True)
    _put_on_path(destination)


_prepare_src()
os.environ["FILAMENT_DATA_ROOT"] = TEST_IMAGES

from filament.baseline import iter_filament_counts, segment_filaments
from filament.runtime import DataRootError, resolve_runtime
from filament.submission import filament_id, write_submission

runtime = resolve_runtime()
print(
    f"profile={runtime.profile} device={runtime.device} output_dir={runtime.output_dir}"
)

## Synthetic demo

One dark rectangle on a bright 64×64 field. Same geometry as `tests/test_baseline_smoke.py`. No images are read here. The arguments are the small ones from that test; the full-disk defaults are for frames on the order of 2k pixels.


In [ ]:
demo = np.full((64, 64), 220, dtype=np.uint8)
demo[24:40, 22:42] = 30
demo_labels = segment_filaments(demo, block_size=31, min_area=16, offset=5)
demo_rows = [
    (filament_id("synthetic64", number), counts)
    for number, counts in iter_filament_counts(demo_labels)
]
print(f"synthetic 64x64 demo: {len(demo_rows)} component(s)")
for row_id, counts in demo_rows:
    print(f"{row_id}: {len(counts)} RLE characters")

## Test images under `FILAMENT_DATA_ROOT`

Skipped when that folder is not a directory. The path is `test/test_images`, not `train`. Files are `.jpeg`. One label image is built per file, then each component is encoded to RLE and written with `write_submission`. Full-size masks are not collected.


In [ ]:
IMAGE_SUFFIXES = {".png", ".jpg", ".jpeg", ".tif", ".tiff"}


def _read_grayscale(path: Path) -> np.ndarray:
    from skimage.color import rgb2gray
    from skimage.io import imread

    loaded = np.asarray(imread(path))
    if loaded.ndim == 3 and loaded.shape[-1] == 1:
        loaded = loaded[..., 0]
    if loaded.ndim == 2:
        return loaded
    if loaded.ndim == 3 and loaded.shape[-1] in (3, 4):
        return rgb2gray(loaded[..., :3])
    raise ValueError(f"expected a grayscale image, got shape {loaded.shape}")


def _iter_rows(data_root: Path):
    files = sorted(path for path in data_root.rglob("*") if path.is_file())
    for path in files:
        rel = path.relative_to(data_root)
        if path.suffix.lower() not in IMAGE_SUFFIXES:
            print(f"skip {rel}: not a test image")
            continue
        try:
            image = _read_grayscale(path)
            labels = segment_filaments(image)
        except (OSError, ValueError) as exc:
            print(f"skip {rel}: {exc}")
            continue
        stem = "_".join(rel.with_suffix("").parts)
        count = 0
        for number, counts in iter_filament_counts(labels):
            count += 1
            yield (filament_id(stem, number), counts)
        del labels
        print(f"{rel}: {count} component(s)")


def _write_rows(data_root: Path) -> None:
    written = 0

    def _counted_rows():
        nonlocal written
        for row in _iter_rows(data_root):
            written += 1
            yield row

    runtime.output_dir.mkdir(parents=True, exist_ok=True)
    destination = runtime.output_dir / "submission.csv"
    write_submission(destination, _counted_rows())
    print(f"wrote {written} row(s) to {destination}")


try:
    data_root = runtime.require_data_root()
except DataRootError as exc:
    print(exc)
else:
    if not data_root.is_dir():
        print(
            f"Data root {data_root} is not a directory. "
            "Synthetic demo only; no submission.csv."
        )
    else:
        _write_rows(data_root)